# Olist E-Commerce Analytics: From Raw Data to Executive Dashboard
**Tech Stack:** Python | PostgreSQL | Power BI

## 🎯 Goal of this Notebook
Turn the nine raw Olist CSV files into clean, correctly typed tables and load them into a PostgreSQL database, so they are ready for SQL analysis and Power BI reporting.

## 🛠️ What this Notebook Does

* **Loads** all 9 CSVs straight from the zip archive.
* **Checks** each table for duplicates, nulls, and incorrect data types.
* **Fixes types:** 
  * IDs become strings.
  * Zip codes are zero-padded to 5 digits.
  * Dates become datetimes.
  * Money columns are rounded to 2 decimals.
* **Reduces** geolocation data from ~1M rows to 19,015 by averaging latitude and longitude per zip prefix.
* **Fills** missing product categories with "Unknown" and adds English category names (including manual translations for the two categories missing from the translation file).
* **Engineers** new features on the `orders` table: 
  * `delivery_time_days`
  * `target_delivery_time_days`
  * `is_late`
* **Defines** explicit SQL `dtypes` and pushes the 9 cleaned tables to the `ecommerce_database` PostgreSQL database.

In [1]:
import pandas as pd
import zipfile
from sqlalchemy import create_engine, text
from sqlalchemy.types import Text, Integer, Numeric, Float, DateTime, Boolean

In [2]:
raw_tables = {}
# Open the zip and list what's inside
with zipfile.ZipFile('archive (1).zip', 'r') as z:
    for file_name in z.namelist():
        if file_name.endswith('.csv'):
            with z.open(file_name) as file:
                raw_tables[file_name] = pd.read_csv(file)
                
print("Data loaded successfully! Here are your files:\n")
print(raw_tables.keys())

Data loaded successfully! Here are your files:

dict_keys(['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv'])


In [3]:
# dictionary hold cleaned dataframes
cleaned_tables = {}

In [4]:
sql_dtypes = {}

# Customers Dataset

In [5]:
customers = raw_tables['olist_customers_dataset.csv'].copy() 
# .copy() enusures we don't mess up the original loaded data

In [6]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [7]:
customers.duplicated().sum()

np.int64(0)

In [8]:
customers.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [9]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [10]:
customers['customer_id'] = customers['customer_id'].astype('string')
customers['customer_unique_id'] = customers['customer_unique_id'].astype('string')
customers['customer_zip_code_prefix'] = (
    customers['customer_zip_code_prefix'].astype('string').str.zfill(5)
)
customers['customer_city'] = customers['customer_city'].astype('string')
customers['customer_state'] = customers['customer_state'].astype('string')

In [11]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  string
 1   customer_unique_id        99441 non-null  string
 2   customer_zip_code_prefix  99441 non-null  string
 3   customer_city             99441 non-null  string
 4   customer_state            99441 non-null  string
dtypes: string(5)
memory usage: 3.8 MB


In [12]:
sql_dtypes['customers'] = {
    'customer_id': Text, 'customer_unique_id': Text, 
    'customer_zip_code_prefix': Text,
    'customer_city': Text, 'customer_state': Text,
}

In [13]:
cleaned_tables['customers'] = customers

# Geolocation dataset

In [14]:
geolocation = raw_tables['olist_geolocation_dataset.csv'].copy()

In [15]:
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [16]:
geolocation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  object 
 4   geolocation_state            1000163 non-null  object 
dtypes: float64(2), int64(1), object(2)
memory usage: 38.2+ MB


In [17]:
geolocation['geolocation_zip_code_prefix'] = (
    geolocation['geolocation_zip_code_prefix'].astype('string').str.zfill(5)
)
geolocation['geolocation_lat'] = geolocation['geolocation_lat'].astype('float64')
geolocation['geolocation_lng'] = geolocation['geolocation_lng'].astype('float64')
geolocation['geolocation_city'] = geolocation['geolocation_city'].astype('string')
geolocation['geolocation_state'] = geolocation['geolocation_state'].astype('string')

In [18]:
geolocation.isna().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [19]:
geolocation.duplicated().sum()

np.int64(261831)

In [20]:
geolocation['geolocation_zip_code_prefix'].is_unique

False

In [21]:
geolocation = geolocation.drop_duplicates()

geolocation = (
    geolocation
    .groupby('geolocation_zip_code_prefix', as_index=False)
    .agg(
        geolocation_lat=('geolocation_lat', 'mean'), 
        geolocation_lng=('geolocation_lng', 'mean'), 
        geolocation_city=('geolocation_city', 'first'),
        geolocation_state=('geolocation_state', 'first'),
    )
)

In [22]:
geolocation['geolocation_zip_code_prefix'].is_unique

True

In [23]:
geolocation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19015 entries, 0 to 19014
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   geolocation_zip_code_prefix  19015 non-null  string 
 1   geolocation_lat              19015 non-null  float64
 2   geolocation_lng              19015 non-null  float64
 3   geolocation_city             19015 non-null  string 
 4   geolocation_state            19015 non-null  string 
dtypes: float64(2), string(3)
memory usage: 742.9 KB


In [24]:
sql_dtypes['geolocation'] = {
    'geolocation_zip_code_prefix': Text,
    'geolocation_lat': Float, 'geolocation_lng': Float,
    'geolocation_city': Text, 'geolocation_state': Text,
}

In [25]:
cleaned_tables['geolocation'] = geolocation

# Order items dataset

In [26]:
order_items = raw_tables['olist_order_items_dataset.csv'].copy()

In [27]:
pd.concat([order_items.head(), order_items.tail()])

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72
112649,fffe41c64501cc87c801fd61db3f6244,1,350688d9dc1e75ff97be326363655e01,f7ccf836d21b2fb1de37564105216cc1,2018-06-12 17:10:13,43.00,12.79


In [28]:
order_items.describe().T

,count,mean,std,min,25%,50%,75%,max
order_item_id,112650.0,1.197834,0.705124,1.00,1.00,1.00,1.00,21.00
price,112650.0,120.653739,183.633928,0.85,39.90,74.99,134.90,6735.00
freight_value,112650.0,19.990320,15.806405,0.00,13.08,16.26,21.15,409.68


In [29]:
order_items.duplicated().sum()

np.int64(0)

In [30]:
order_items.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [31]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [32]:
order_items['order_id'] = order_items['order_id'].astype('string')
order_items['order_item_id'] = order_items['order_item_id'].astype('int8')
order_items['product_id'] = order_items['product_id'].astype('string')
order_items['seller_id'] = order_items['seller_id'].astype('string')
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

In [33]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  string        
 1   order_item_id        112650 non-null  int8          
 2   product_id           112650 non-null  string        
 3   seller_id            112650 non-null  string        
 4   shipping_limit_date  112650 non-null  datetime64[ns]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int8(1), string(3)
memory usage: 5.3 MB


In [34]:
sql_dtypes['order_items'] = {
    'order_id': Text, 'order_item_id': Integer, 
    'product_id': Text, 'seller_id': Text, 
    'shipping_limit_date': DateTime,
    'price': Numeric(10, 2), 'freight_value': Numeric(10,2),
}

In [35]:
cleaned_tables['order_items'] = order_items

In [36]:
cleaned_tables['order_items'][['price', 'freight_value']] = (
    cleaned_tables['order_items'][['price', 'freight_value']].round(2)
)

# Order payments dataset

In [37]:
order_payment = raw_tables['olist_order_payments_dataset.csv'].copy()

In [38]:
order_payment.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [39]:
order_payment.describe().T

,count,mean,std,min,25%,50%,75%,max
payment_sequential,103886.0,1.092679,0.706584,1.0,1.00,1.0,1.0000,29.00
payment_installments,103886.0,2.853349,2.687051,0.0,1.00,1.0,4.0000,24.00
payment_value,103886.0,154.100380,217.494064,0.0,56.79,100.0,171.8375,13664.08


In [40]:
order_payment.duplicated().sum()

np.int64(0)

In [41]:
order_payment.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [42]:
order_payment.payment_type.value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [43]:
order_payment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


In [44]:
order_payment['order_id'] = order_payment['order_id'].astype('string')
order_payment['payment_sequential'] = order_payment['payment_sequential'].astype('int8')
order_payment['payment_type'] = order_payment['payment_type'].astype('category')
order_payment['payment_installments'] = order_payment['payment_installments'].astype('int8')

In [45]:
order_payment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype   
---  ------                --------------   -----   
 0   order_id              103886 non-null  string  
 1   payment_sequential    103886 non-null  int8    
 2   payment_type          103886 non-null  category
 3   payment_installments  103886 non-null  int8    
 4   payment_value         103886 non-null  float64 
dtypes: category(1), float64(1), int8(2), string(1)
memory usage: 1.9 MB


In [46]:
sql_dtypes['order_payment'] = {
    'order_id': Text, 'payment_sequential': Integer,
    'payment_type': Text, 'payment_installments': Integer,
    'payment_value': Numeric(12, 2),
}

In [47]:
cleaned_tables['order_payment'] = order_payment

In [48]:
cleaned_tables['order_payment']['payment_value'] = (
    cleaned_tables['order_payment']['payment_value'].round(2)
)

# Order review dataset

In [49]:
order_reviews = raw_tables['olist_order_reviews_dataset.csv'].copy()

In [50]:
order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [51]:
order_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [52]:
order_reviews.describe().T

,count,mean,std,min,25%,50%,75%,max
review_score,99224.0,4.086421,1.347579,1.0,4.0,5.0,5.0,5.0


In [53]:
order_reviews.duplicated().sum()

np.int64(0)

In [54]:
order_reviews.review_score.value_counts()

review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64

In [55]:
order_reviews['review_id'] = order_reviews['review_id'].astype('string')
order_reviews['order_id'] = order_reviews['order_id'].astype('string')
order_reviews['review_score'] = order_reviews['review_score'].astype('int8')
order_reviews['review_comment_title'] = order_reviews['review_comment_title'].astype('string')
order_reviews['review_comment_message'] = order_reviews['review_comment_message'].astype('string')
order_reviews['review_creation_date'] = pd.to_datetime(order_reviews['review_creation_date'])
order_reviews['review_answer_timestamp'] = pd.to_datetime(order_reviews['review_answer_timestamp'])

In [56]:
order_reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [57]:
#pd.set_option('display.max_rows', 500)

In [58]:
pd.reset_option('display.max_rows')

In [59]:
order_reviews[['review_comment_title', 'review_comment_message']].head(500)

,review_comment_title,review_comment_message
0,<NA>,<NA>
1,<NA>,<NA>
2,<NA>,<NA>
3,<NA>,Recebi bem antes do prazo estipulado.
4,<NA>,Parabéns lojas lannister adorei comprar pela I...
...,...,...
495,Ótimo,<NA>
496,<NA>,Pensei que a mochila fosse um material mais re...
497,<NA>,<NA>
498,<NA>,<NA>


In [60]:
order_reviews['review_comment_title'] = order_reviews['review_comment_title'].fillna('Unknown')
order_reviews['review_comment_message'] = order_reviews['review_comment_message'].fillna('Unknown')

In [61]:
order_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  string        
 1   order_id                 99224 non-null  string        
 2   review_score             99224 non-null  int8          
 3   review_comment_title     99224 non-null  string        
 4   review_comment_message   99224 non-null  string        
 5   review_creation_date     99224 non-null  datetime64[ns]
 6   review_answer_timestamp  99224 non-null  datetime64[ns]
dtypes: datetime64[ns](2), int8(1), string(4)
memory usage: 4.6 MB


In [62]:
sql_dtypes['order_reviews'] = {
    'review_id': Text, 'order_id': Text, 'review_score': Integer,
    'review_comment_title': Text, 'review_comment_message': Text,
    'review_creation_date': DateTime, 'review_answer_timestamp': DateTime,
}

In [63]:
cleaned_tables['order_reviews'] = order_reviews

# Orders dataset

In [64]:
orders_dataset = raw_tables['olist_orders_dataset.csv'].copy()

In [65]:
orders_dataset.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


To calculate the acutal delivery days (the total time it took for the customer to receive their item), you should find the difference between these two columns:
* Order_purchase_timestamp (when the order was initially placed)
* Order_delivered_customer_date (when the item actually reached the customer)

If you are looking to calculate the projected delivery time that the company promised the customer at checkout, you should instead use _order_purchase_timestamp_ and _order_estimated_delivery_date_. 

In [66]:
orders_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [67]:
orders_dataset['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [68]:
orders_dataset['order_id'] = orders_dataset['order_id'].astype('string')
orders_dataset['customer_id'] = orders_dataset['customer_id'].astype('string')
orders_dataset['order_status'] = orders_dataset['order_status'].astype('category')
orders_dataset['order_purchase_timestamp'] = (
                pd.to_datetime(orders_dataset['order_purchase_timestamp'])
)
orders_dataset['order_approved_at'] = (
                pd.to_datetime(orders_dataset['order_approved_at'])
)
orders_dataset['order_delivered_carrier_date'] = (
                pd.to_datetime(orders_dataset['order_delivered_carrier_date'])
)
orders_dataset['order_delivered_customer_date'] = (
                pd.to_datetime(orders_dataset['order_delivered_customer_date'])
)
orders_dataset['order_estimated_delivery_date'] = (
                pd.to_datetime(orders_dataset['order_estimated_delivery_date'])
)

In [69]:
# Create a column for 'Delivery Duration' in days
orders_dataset['delivery_time_days'] = (
    orders_dataset['order_delivered_customer_date'] - orders_dataset['order_purchase_timestamp']
).dt.days

orders_dataset['target_delivery_time_days'] = (
    orders_dataset['order_estimated_delivery_date'] - orders_dataset['order_purchase_timestamp']
).dt.days

In [70]:
orders_dataset.describe().T

,count,mean,min,25%,50%,75%,max,std
order_purchase_timestamp,99441,2017-12-31 08:43:12.776581120,2016-09-04 21:15:19,2017-09-12 14:46:19,2018-01-18 23:04:36,2018-05-04 15:42:16,2018-10-17 17:30:18,NaN
order_approved_at,99281,2017-12-31 18:35:24.098800128,2016-09-15 12:16:38,2017-09-12 23:24:16,2018-01-19 11:36:13,2018-05-04 20:35:10,2018-09-03 17:40:06,NaN
order_delivered_carrier_date,97658,2018-01-04 21:49:48.138278656,2016-10-08 10:34:01,2017-09-15 22:28:50.249999872,2018-01-24 16:10:58,2018-05-08 13:37:45,2018-09-11 19:48:28,NaN
order_delivered_customer_date,96476,2018-01-14 12:09:19.035542272,2016-10-11 13:46:32,2017-09-25 22:07:22.249999872,2018-02-02 19:28:10.500000,2018-05-15 22:48:52.249999872,2018-10-17 13:22:46,NaN
order_estimated_delivery_date,99441,2018-01-24 03:08:37.730111232,2016-09-30 00:00:00,2017-10-03 00:00:00,2018-02-15 00:00:00,2018-05-25 00:00:00,2018-11-12 00:00:00,NaN
delivery_time_days,96476.0,12.094086,0.0,6.0,10.0,15.0,209.0,9.551746
target_delivery_time_days,99441.0,23.403958,1.0,18.0,23.0,28.0,155.0,8.829562


In [71]:
# Convert the float column to a nullable integer column
orders_dataset['delivery_time_days'] = orders_dataset['delivery_time_days'].astype('Int16')
orders_dataset['target_delivery_time_days'] = (
    orders_dataset['target_delivery_time_days'].astype('Int16')
)

In [72]:
orders_dataset['is_late'] = (
    orders_dataset['delivery_time_days'] > orders_dataset['target_delivery_time_days']
)

In [73]:
orders_dataset.is_late.value_counts()

is_late
False    89168
True      7308
Name: count, dtype: Int64

In [74]:
orders_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  string        
 1   customer_id                    99441 non-null  string        
 2   order_status                   99441 non-null  category      
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
 8   delivery_time_days             96476 non-null  Int16         
 9   target_delivery_time_days      99441 non-null  Int16         
 10  is_late                        96476 non-null  boolean       
dtypes: Int16(2), bo

In [75]:
sql_dtypes['orders'] = {
    'order_id': Text, 'customer_id': Text, 'order_status': Text,
    'order_purchase_timestamp': DateTime,
    'order_approved_at': DateTime,
    'order_delivered_carrier_date': DateTime,
    'order_delivered_customer_date': DateTime,
    'order_estimated_delivery_date': DateTime,
    'delivery_time_days': Integer, 'target_delivery_time_days': Integer,
    'is_late': Boolean,
}

In [76]:
cleaned_tables['orders'] = orders_dataset

# Products dataset

In [77]:
raw_tables.keys()

dict_keys(['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv'])

In [78]:
products = raw_tables['olist_products_dataset.csv'].copy()

In [79]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [80]:
products.duplicated().sum()

np.int64(0)

In [81]:
products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [82]:
products[products["product_category_name"].isnull()]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


In [83]:
# Fill missing categories with 'Unknown'
products['product_category_name'] = products['product_category_name'].fillna('Unknown')

products.isnull().sum()

product_id                      0
product_category_name           0
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [84]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32951 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [85]:
products['product_id'] = products['product_id'].astype('string')
products['product_category_name'] = products['product_category_name'].astype('string')

In [86]:
products = products.rename(columns={
    'product_name_lenght': 'product_name_length', 
    'product_description_lenght': 'product_description_length'
})

In [87]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  string 
 1   product_category_name       32951 non-null  string 
 2   product_name_length         32341 non-null  float64
 3   product_description_length  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), string(2)
memory usage: 2.3 MB


In [88]:
products.describe().T

,count,mean,std,min,25%,50%,75%,max
product_name_length,32341.0,48.476949,10.245741,5.0,42.0,51.0,57.0,76.0
product_description_length,32341.0,771.495285,635.115225,4.0,339.0,595.0,972.0,3992.0
product_photos_qty,32341.0,2.188986,1.736766,1.0,1.0,1.0,3.0,20.0
product_weight_g,32949.0,2276.472488,4282.038731,0.0,300.0,700.0,1900.0,40425.0
product_length_cm,32949.0,30.815078,16.914458,7.0,18.0,25.0,38.0,105.0
product_height_cm,32949.0,16.937661,13.637554,2.0,8.0,13.0,21.0,105.0
product_width_cm,32949.0,23.196728,12.079047,6.0,15.0,20.0,30.0,118.0


In [89]:
cleaned_tables['products'] = products

# Product category name translation

In [90]:
category_name_translation = raw_tables['product_category_name_translation.csv'].copy()

In [91]:
category_name_translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [92]:
category_name_translation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     object
 1   product_category_name_english  71 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB


In [93]:
category_name_translation['product_category_name'] = (
    category_name_translation['product_category_name'].astype('string')
)
category_name_translation['product_category_name_english'] = (
    category_name_translation['product_category_name_english'].astype('string')
)

In [94]:
category_name_translation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     string
 1   product_category_name_english  71 non-null     string
dtypes: string(2)
memory usage: 1.2 KB


In [95]:
cleaned_tables['category_name_translation'] = category_name_translation

# Adding the product category name translation column

In [96]:
products = cleaned_tables['products']
translation = cleaned_tables['category_name_translation']

In [97]:
missing = set(products['product_category_name']) - set(translation['product_category_name'])
print(missing)

{'portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer', 'Unknown'}


In [98]:
extra = pd.DataFrame({
    'product_category_name': ['portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer', 'Unknown'],
    'product_category_name_english': ['portable_kitchen_food_preparers', 'pc_gamer', 'Uknown'], 
}).astype('string')

In [99]:
translation = pd.concat([translation, extra], ignore_index=True)

In [100]:
products = products.merge(translation, on='product_category_name', how='left')

In [101]:
print(len(products))

32951


In [102]:
print(products['product_category_name_english'].isna().sum())

0


In [103]:
cleaned_tables['products'] = products
cleaned_tables['category_name_translation'] = translation

In [104]:
products.head()

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


In [119]:
sql_dtypes['products'] = {
    'product_id': Text, 'product_category_name': Text,
    'product_category_name_english': Text,
    'product_name_length': Float, 'product_description_length': Float,
    'product_photos_qty': Float,
    'product_weight_g': Float, 'product_length_cm': Float,
    'product_height_cm': Float, 'product_width_cm': Float,
}

In [106]:
sql_dtypes['category_name_translation'] = {
    'product_category_name': Text, 'product_category_name_english': Text,
}

# Sellers dataset

In [107]:
sellers = raw_tables['olist_sellers_dataset.csv'].copy()

In [108]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [109]:
sellers.duplicated().sum()

np.int64(0)

In [110]:
sellers.isnull().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [111]:
sellers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   object
 1   seller_zip_code_prefix  3095 non-null   int64 
 2   seller_city             3095 non-null   object
 3   seller_state            3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB


In [112]:
sellers['seller_id'] = sellers['seller_id'].astype('string')
sellers['seller_zip_code_prefix'] = (
    sellers['seller_zip_code_prefix'].astype('string').str.zfill(5)
)
sellers['seller_city'] = sellers['seller_city'].astype('string')
sellers['seller_state'] = sellers['seller_state'].astype('string')

In [113]:
sql_dtypes['sellers'] = {
    'seller_id': Text, 'seller_zip_code_prefix': Text,
    'seller_city': Text, 'seller_state': Text,
}

In [114]:
cleaned_tables['sellers'] = sellers

# Pushing all the files to SQL

In [120]:
for name, df in cleaned_tables.items():
    print(name, set(sql_dtypes[name]) - set(df.columns))   # should print set() for each

customers set()
geolocation set()
order_items set()
order_payment set()
order_reviews set()
orders set()
products set()
category_name_translation set()
sellers set()


In [121]:
# Connect to the default 'postgres' database with AUTOCOMMIT
default_engine = create_engine(
    "postgresql+psycopg2://postgres@localhost:5432/postgres", 
    isolation_level="AUTOCOMMIT"
)

# Create the new database
with default_engine.connect() as conn:
    # We use a try-except block just in case you run this cell twice
    try:
        conn.execute(text("CREATE DATABASE ecommerce_database;"))
        print("Database created successfully!")
    except Exception as e:
        print("Database already exists or error:", e)

Database already exists or error: (psycopg2.errors.DuplicateDatabase) database "ecommerce_database" already exists

[SQL: CREATE DATABASE ecommerce_database;]
(Background on this error at: https://sqlalche.me/e/20/f405)


In [122]:
# Connect to your newly created database using your original string
cnxn_string = "postgresql+psycopg2://postgres@localhost:5432/ecommerce_database"
engine = create_engine(cnxn_string)

In [123]:
for name, df in cleaned_tables.items():
    df.to_sql(name, engine, if_exists='replace', index=False,
              dtype=sql_dtypes[name], chunksize=50000)